# Restaurant Analytics Agent - Data Foundation

This notebook creates Unity Catalog tables that power a ReAct-pattern agent with four tools:
- **Business Lookup**: Retrieve restaurant ratings, review counts, and metadata
- **Vector Search**: Semantic search over customer reviews (requires review text embeddings)
- **Competitor Comparison**: Compare ratings and trends across restaurants
- **Complaint/Opportunity Extraction**: Identify recurring themes in reviews

**Data pipeline**: Ingest compressed Google Local data → filter San Diego Italian restaurants → create Delta tables in Unity Catalog

**Agent stack**: 
- LLM: Meta Llama 3.3 70B Instruct (Databricks Foundation Model API)
- Embeddings: GTE Large (En) for vector search
- Framework: LangGraph for tool routing

In [0]:
# Install required packages
%pip install databricks-vectorsearch
%pip install databricks-vectorsearch openai
dbutils.library.restartPython()

Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 33.2 MB/s eta 0:00:00
Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.


---
## 1. Setup Unity Catalog Structure

Create the catalog, schema, and volume for staging source data and storing agent tables.

**UC namespace**: `main.aai510_final_agent`  
**Volume**: Stores compressed source files for reusable Spark reads (no local `wget` downloads)  
**Source data**:
- `meta-California.json.gz` - Business metadata (name, address, categories, location)
- `rating-California.csv.gz` - User ratings (gmap_id, user_id, rating, timestamp)

In [0]:
from pyspark.sql import functions as F

# Configuration
CATALOG = "main"
SCHEMA = "aai510_final_agent"
VOLUME = "aai510_final_raw"

# Source URLs
META_URL = "https://mcauleylab.ucsd.edu:8443/public_datasets/gdrive/googlelocal/meta-California.json.gz"
RATING_URL = "https://mcauleylab.ucsd.edu:8443/public_datasets/gdrive/googlelocal/rating-California.csv.gz"

# Create Unity Catalog objects
print("=" * 60)
print("Creating Unity Catalog structure...")
print("=" * 60)

spark.sql(f"CREATE CATALOG IF NOT EXISTS {CATALOG}")
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {CATALOG}.{SCHEMA}")
spark.sql(f"CREATE VOLUME IF NOT EXISTS {CATALOG}.{SCHEMA}.{VOLUME}")

# Volume paths for staged files
META_PATH = f"/Volumes/{CATALOG}/{SCHEMA}/{VOLUME}/meta-California.json.gz"
RATING_PATH = f"/Volumes/{CATALOG}/{SCHEMA}/{VOLUME}/rating-California.csv.gz"

# Helper function to check whether files already exist in Volume
def file_exists(path):
    try:
        dbutils.fs.ls(path)
        return True
    except Exception:
        return False

print(f"\nStaging compressed files to Volume: {CATALOG}.{SCHEMA}.{VOLUME}")
print("Checking for existing staged files...\n")

# Stage files into Volume only if needed
if file_exists(META_PATH):
    print(f"✓ Already staged: {META_PATH}")
else:
    dbutils.fs.cp(META_URL, META_PATH)
    print(f"✓ Staged: {META_PATH}")

if file_exists(RATING_PATH):
    print(f"✓ Already staged: {RATING_PATH}")
else:
    dbutils.fs.cp(RATING_URL, RATING_PATH)
    print(f"✓ Staged: {RATING_PATH}")

print(f"\n✓ Unity Catalog setup complete")
print(f"\tCatalog: {CATALOG}")
print(f"\tSchema: {CATALOG}.{SCHEMA}")
print(f"\tVolume: {CATALOG}.{SCHEMA}.{VOLUME}")

Creating Unity Catalog structure...

Staging compressed files to Volume: main.aai510_final_agent.aai510_final_raw
Checking for existing staged files...

✓ Already staged: /Volumes/main/aai510_final_agent/aai510_final_raw/meta-California.json.gz
✓ Already staged: /Volumes/main/aai510_final_agent/aai510_final_raw/rating-California.csv.gz

✓ Unity Catalog setup complete
	Catalog: main
	Schema: main.aai510_final_agent
	Volume: main.aai510_final_agent.aai510_final_raw


---
### 1.1. Inspect Staged Files

Verify that files were successfully staged and examine their schemas to understand the data structure.

**What we're checking**:
- Total record counts for California dataset
- Schema structure (column names and types)
- Sample records to confirm data quality
- Whether review text is available (needed for Vector Search)

This step helps identify any data quality issues early and confirms which agent tools can be implemented with the current dataset.

In [0]:
# Verify staged files and inspect schema
print("=" * 60)
print("Inspecting Staged Files...")
print("=" * 60)

# Read metadata sample
print("\n1. Business Metadata (meta-California.json.gz)")
print("-" * 60)
meta_df = spark.read.json(META_PATH)
print(f"Total businesses in California: {meta_df.count():,}")
print("\nSample records:")
display(meta_df.limit(3))

# Read ratings sample
print("\n2. Ratings Data (rating-California.csv.gz)")
print("-" * 60)
ratings_df = spark.read.option("header", "true").csv(RATING_PATH)
print(f"Total ratings in California: {ratings_df.count():,}")
print("\nSample records:")
display(ratings_df.limit(3))

Inspecting Staged Files...

1. Business Metadata (meta-California.json.gz)
------------------------------------------------------------
Total businesses in California: 515,961

Sample records:


MISC,address,avg_rating,category,description,gmap_id,hours,latitude,longitude,name,num_of_reviews,price,relative_results,state,url
null,"City Textile, 3001 E Pico Blvd, Los Angeles, CA 90023",4.5,List(Textile exporter),null,0x80c2c98c0e3c16fd:0x29ec8a728764fdf9,null,34.0188913,-118.2152898,City Textile,6,null,"List(0x80c2c624136ea88b:0xb0315367ed448771, 0x80c2c97cb7c52f17:0xb66ee68c1c366f2d)",Open now,https://www.google.com/maps/place//data=!4m2!3m1!1s0x80c2c98c0e3c16fd:0x29ec8a728764fdf9?authuser=-1&hl=en&gl=us
"List(List(Wheelchair accessible entrance), null, List(Good for kids), List(Casual), null, null, null, null, null, null, null, null, List(Comfort food), null, null, null, null, List(Takeout, Dine-in, Delivery))","San Soo Dang, 761 S Vermont Ave, Los Angeles, CA 90005",4.4,List(Korean restaurant),null,0x80c2c778e3b73d33:0xbdc58662a4a97d49,"List(List(Thursday, 6:30AM–6PM), List(Friday, 6:30AM–6PM), List(Saturday, 6:30AM–6PM), List(Sunday, 7AM–12PM), List(Monday, Closed), List(Tuesday, 6:30AM–6PM), List(Wednesday, 6:30AM–6PM))",34.0580917,-118.2921295,San Soo Dang,18,null,"List(0x80c2c78249aba68f:0x35bf16ce61be751d, 0x80c2c79998f75fff:0xd7ca5c67e96fb778, 0x80c2b899146d7507:0xf4162c12c9cf65f8, 0x80c2c77f2d419951:0x26285631b21e324c, 0x80c2b8add9016015:0x15836f81a963b35f)",Open ⋅ Closes 6PM,https://www.google.com/maps/place//data=!4m2!3m1!1s0x80c2c778e3b73d33:0xbdc58662a4a97d49?authuser=-1&hl=en&gl=us
"List(null, null, null, null, null, null, null, null, null, null, null, null, null, List(Checks, Debit cards, Credit cards), null, null, null, List(In-store shopping))","Nova Fabrics, 2200 E 11th St, Los Angeles, CA 90021",3.3,List(Fabric store),null,0x80c2c89923b27a41:0x32041559418d447,"List(List(Thursday, 9AM–5PM), List(Friday, 9AM–5PM), List(Saturday, Closed), List(Sunday, Closed), List(Monday, 9AM–5PM), List(Tuesday, 9AM–5PM), List(Wednesday, 9AM–5PM))",34.0236689,-118.2329297,Nova Fabrics,6,null,"List(0x80c2c8811477253f:0x23a8a492df1918f7, 0x80c2c6333ea5aaa7:0x38c93218b0d5751b, 0x80c2c633235effa1:0x384bf96fe70cbbc7, 0x80c2c6295991d031:0x8516999f61103f87, 0x80c2c63331db7ec5:0x507f59d850d36691)",Open ⋅ Closes 5PM,https://www.google.com/maps/place//data=!4m2!3m1!1s0x80c2c89923b27a41:0x32041559418d447?authuser=-1&hl=en&gl=us



2. Ratings Data (rating-California.csv.gz)
------------------------------------------------------------
Total ratings in California: 70,082,062

Sample records:


business,user,rating,timestamp
0x80c2c98c0e3c16fd:0x29ec8a728764fdf9,113165551130476225599,5,1599164133778
0x80c2c98c0e3c16fd:0x29ec8a728764fdf9,101226371370637614545,5,1618261672851
0x80c2c98c0e3c16fd:0x29ec8a728764fdf9,111167703666981482712,5,1524515066787


---
## 2. Filter San Diego Italian Restaurants

Read business metadata and filter for San Diego Italian restaurants using Spark SQL expressions. This creates the foundational `restaurants` table that powers the Business Lookup tool and defines which reviews are relevant for analysis.

**Filters applied**:
- **Geography**: Address contains "San Diego, CA"
- **Category**: Business category includes "Restaurant" AND "Italian"

**Output table**: `main.aai510_final_agent.restaurants`

In [0]:
# Read and filter business metadata
print("=" * 60)
print("Filtering San Diego Italian restaurants...")
print("=" * 60)

# Read metadata with schema inference
meta_df = spark.read.json(META_PATH)

# Filter logic (mimics original regex + category matching)
restaurants_df = (
    meta_df
    # Convert category array to searchable string
    .withColumn("category_text", 
                F.lower(F.concat_ws(" ", F.coalesce(F.col("category"), F.array()))))
    # Normalize address for matching
    .withColumn("address_lower", 
                F.lower(F.coalesce(F.col("address"), F.lit(""))))
    # Apply filters
    .filter(F.col("address_lower").rlike(r",\s*san diego,\s*ca\b"))
    .filter(F.col("category_text").contains("restaurant"))
    .filter(F.col("category_text").contains("italian"))
    # Select clean schema for agent tools
    .select(
        "gmap_id",
        F.col("name").alias("business_name"),
        "address",
        "category",
        "latitude",
        "longitude"
    )
    .dropna(subset=["gmap_id"])
    .dropDuplicates(["gmap_id"])
)

# Preview results
restaurant_count = restaurants_df.count()
print(f"\n✓ Found {restaurant_count:,} San Diego Italian restaurants")

print("\nSample restaurants:")
display(restaurants_df.limit(5))

# Write to Unity Catalog
table_name = f"{CATALOG}.{SCHEMA}.restaurants"

print("=" * 60)
print(f"\nWriting to Unity Catalog: {table_name}")
print("=" * 60)

restaurants_df.write.mode("overwrite").option("mergeSchema", "true").saveAsTable(table_name)
print(f"✓ Table created: {table_name}")

Filtering San Diego Italian restaurants...

✓ Found 199 San Diego Italian restaurants

Sample restaurants:


gmap_id,business_name,address,category,latitude,longitude
0x80d9470097a21f59:0xd2b0f26d5e6970f5,Bambino Restaurant,"Bambino Restaurant, 2493 Roll Dr, San Diego, CA 92154","List(Italian restaurant, Pizza restaurant)",32.553638299999996,-116.9372631
0x80d949cf0a02e68b:0x448d43ff67d4b5dd,Sbarro,"Sbarro, 4211 Camino De La Plaza, San Diego, CA 92173","List(Pizza restaurant, Caterer, Fast food restaurant, Italian restaurant, Pasta shop, Pizza delivery, Pizza Takeout)",32.543172399999996,-117.0410819
0x80d94c1102785aff:0x4bbbb21a7aa26d1,hot tasty Pizza,"hot tasty Pizza, 1826 Coronado Ave, San Diego, CA 92154","List(Pizza restaurant, Italian restaurant)",32.5768088,-117.094146
0x80d94c229bbb1cd3:0xae528744f7d32b45,All American Sandwiches N Pizza,"All American Sandwiches N Pizza, 685 Saturn Blvd, San Diego, CA 92154","List(Italian restaurant, Pizza delivery, Pizza restaurant, Pizza Takeout, Restaurant, Sandwich shop)",32.585566,-117.089107
0x80d94eca86188a57:0xa42c358bd1b763fe,Mike's New York Giant Pizza,"Mike's New York Giant Pizza, 1270 Picador Blvd, San Diego, CA 92154","List(Pizza restaurant, Italian restaurant, Delivery Restaurant, Restaurant)",32.5744195,-117.055319



Writing to Unity Catalog: main.aai510_final_agent.restaurants
✓ Table created: main.aai510_final_agent.restaurants


---
## 3. Join Reviews to Filtered Restaurants

Read the ratings CSV and join to our filtered San Diego Italian restaurants. This creates the `reviews` table that contains all customer ratings for our target restaurants.

**Output table**: `main.aai510_final_agent.reviews`  
**Schema**: `gmap_id`, `user_id`, `rating`, `timestamp`

**Note**: This dataset contains ratings only (no review text). For vector search tools, review text will need to be sourced separately.

In [0]:
# Read and join reviews to filtered restaurants
print("=" * 60)
print("Loading and filtering reviews...")
print("=" * 60)

# Read ratings CSV with header
ratings_df = (
    spark.read
    .option("header", "true")
    .csv(RATING_PATH)
    .withColumnRenamed("business", "gmap_id")
    .withColumnRenamed("user", "user_id")
)

print(f"Total ratings in California: {ratings_df.count():,}")

# Load restaurant filter list
restaurants_df = spark.table(f"{CATALOG}.{SCHEMA}.restaurants")
restaurant_ids = restaurants_df.select("gmap_id")

print(f"Filtering to {restaurant_ids.count():,} San Diego Italian restaurants...")

# Inner join to get only reviews for our target restaurants
reviews_df = (
    ratings_df
    .join(restaurant_ids, on="gmap_id", how="inner")
    .select("gmap_id", "user_id", "rating", "timestamp")
)

review_count = reviews_df.count()
print(f"\n✓ Found {review_count:,} reviews for San Diego Italian restaurants")

print("\nSample reviews:")
display(reviews_df.limit(5))

# Write to Unity Catalog
table_name = f"{CATALOG}.{SCHEMA}.reviews"
print(f"\nWriting to Unity Catalog: {table_name}")
reviews_df.write.mode("overwrite").option("mergeSchema", "true").saveAsTable(table_name)
print(f"✓ Table created: {table_name}")

# Summary stats
print("\n" + "=" * 60)
print("SUMMARY")
print("=" * 60)
print(f"Restaurants: {restaurant_ids.count():,}")
print(f"Reviews: {review_count:,}")
print(f"Avg reviews per restaurant: {review_count / restaurant_ids.count():.1f}")

Loading and filtering reviews...
Total ratings in California: 70,082,062
Filtering to 199 San Diego Italian restaurants...

✓ Found 82,845 reviews for San Diego Italian restaurants

Sample reviews:


gmap_id,user_id,rating,timestamp
0x80dc0185c5e2d4df:0xfbc4e830ef1ca431,114976159291119258205,1,1564203402789
0x80dc0185c5e2d4df:0xfbc4e830ef1ca431,100305772822037390573,4,1501468574646
0x80dc0185c5e2d4df:0xfbc4e830ef1ca431,110966308572404885088,5,1552792044271
0x80dc0185c5e2d4df:0xfbc4e830ef1ca431,115555920222063468568,4,1556292850077
0x80dc0185c5e2d4df:0xfbc4e830ef1ca431,101357089882070703342,5,1557093995582



Writing to Unity Catalog: main.aai510_final_agent.reviews
✓ Table created: main.aai510_final_agent.reviews

SUMMARY
Restaurants: 199
Reviews: 82,845
Avg reviews per restaurant: 416.3


---
## 4. Create Restaurant Metrics Table

Aggregate review data to create metrics for each restaurant. This table powers the **Business Lookup** and **Competitor Comparison** agent tools.

**Output table**: `main.aai510_final_agent.restaurant_metrics`  
**Metrics computed**:
- Average rating
- Total review count
- Rating distribution (count by 1-5 stars)
- Latest review timestamp
- Restaurant metadata (name, address, location)

In [0]:
# Create aggregated metrics from reviews
print("=" * 60)
print("Computing restaurant metrics...")
print("=" * 60)

# Load reviews and restaurants
reviews_df = spark.table(f"{CATALOG}.{SCHEMA}.reviews")
restaurants_df = spark.table(f"{CATALOG}.{SCHEMA}.restaurants")

# Aggregate metrics per restaurant
metrics_df = (
    reviews_df
    .groupBy("gmap_id")
    .agg(
        F.avg("rating").alias("avg_rating"),
        F.count("*").alias("review_count"),
        F.max("timestamp").alias("latest_review_timestamp"),
        # Rating distribution: count by rating value (1-5 stars)
        F.sum(F.when(F.col("rating") == "1", 1).otherwise(0)).alias("rating_1_count"),
        F.sum(F.when(F.col("rating") == "2", 1).otherwise(0)).alias("rating_2_count"),
        F.sum(F.when(F.col("rating") == "3", 1).otherwise(0)).alias("rating_3_count"),
        F.sum(F.when(F.col("rating") == "4", 1).otherwise(0)).alias("rating_4_count"),
        F.sum(F.when(F.col("rating") == "5", 1).otherwise(0)).alias("rating_5_count"),
    )
)

# Join with restaurant metadata
restaurant_metrics_df = (
    metrics_df
    .join(restaurants_df, on="gmap_id", how="inner")
    .select(
        "gmap_id",
        "business_name",
        "address",
        "category",
        "latitude",
        "longitude",
        "avg_rating",
        "review_count",
        "latest_review_timestamp",
        "rating_1_count",
        "rating_2_count",
        "rating_3_count",
        "rating_4_count",
        "rating_5_count",
    )
    .orderBy(F.desc("review_count"))
)

print(f"\n✓ Computed metrics for {restaurant_metrics_df.count():,} restaurants")

print("\nTop 10 restaurants by review count:")
display(restaurant_metrics_df.limit(10))

# Write to Unity Catalog
table_name = f"{CATALOG}.{SCHEMA}.restaurant_metrics"
print(f"\nWriting to Unity Catalog: {table_name}")
restaurant_metrics_df.write.mode("overwrite").option("mergeSchema", "true").saveAsTable(table_name)
print(f"✓ Table created: {table_name}")

print("\n" + "=" * 60)
print("✓ Restaurant metrics table ready for agent tools")
print("=" * 60)

Computing restaurant metrics...

✓ Computed metrics for 199 restaurants

Top 10 restaurants by review count:


gmap_id,business_name,address,category,latitude,longitude,avg_rating,review_count,latest_review_timestamp,rating_1_count,rating_2_count,rating_3_count,rating_4_count,rating_5_count
0x80d954b2060ef165:0xc0de3f7db480b86,Filippi's Pizza Grotto Little Italy,"Filippi's Pizza Grotto Little Italy, 1747 India St, San Diego, CA 92101","List(Italian restaurant, Caterer, Pizza restaurant, Sandwich shop, Wine bar)",32.7237596,-117.1681514,4.483330194010171,5309,1618418280662,162,131,352,998,3666
0x80d954d0c6b6ef1f:0xd1f0c147aed157b4,Bronx Pizza,"Bronx Pizza, 111 Washington St, San Diego, CA 92103","List(Pizza restaurant, Italian restaurant, Restaurant)",32.7497524,-117.1634461,4.735729386892178,3311,1619755100238,40,34,93,427,2717
0x80d9535a475211fb:0x16d2e94fee4a042f,The Old Spaghetti Factory,"The Old Spaghetti Factory, 275 Fifth Ave, San Diego, CA 92101","List(Italian restaurant, Banquet hall, Bar, Delivery service, Family restaurant, Gluten-free restaurant, Takeout Restaurant, Restaurant)",32.708209,-117.15995199999999,4.318061088977424,3012,1618800096527,99,110,291,746,1766
0x80d954b3b5f8140f:0xd692822c3ed8379c,Mona Lisa Italian Foods,"Mona Lisa Italian Foods, 2061 India St, San Diego, CA 92101","List(Italian restaurant, Deli, Family restaurant, Pizza restaurant, Restaurant, Wine store)",32.7262337,-117.16907459999999,4.58653149891383,2762,1621915018849,58,52,131,492,2029
0x80d9537e305d3911:0x2108857782e4a0ce,Buona Forchetta - South Park,"Buona Forchetta - South Park, 3001 Beech St, San Diego, CA 92102","List(Pizza restaurant, Italian restaurant, Pasta shop, Restaurant)",32.721163499999996,-117.13003859999999,4.668550873586844,1946,1619496062516,33,35,72,264,1542
0x80deaae4ea5276a3:0x5406b47f1937c678,Olive Garden Italian Restaurant,"Olive Garden Italian Restaurant, 3215 Sports Arena Blvd, San Diego, CA 92110","List(Italian restaurant, Caterer, Family restaurant, Gluten-free restaurant, Takeout Restaurant, Seafood restaurant, Soup restaurant, Wine bar)",32.752378,-117.20728799999999,4.299789251844047,1898,1619048951164,87,64,163,463,1121
0x80d954b21a99cba9:0x76e8c703d80d71f4,Civico 1845,"Civico 1845, 1845 India St, San Diego, CA 92101",List(Italian restaurant),32.724289999999996,-117.168351,4.458424507658643,1828,1619639046264,62,55,112,353,1246
0x80d954ade7b8cbe7:0x65851f19969f4d14,Buon Appetito Restaurant,"Buon Appetito Restaurant, 1609 India St, San Diego, CA 92101","List(Italian restaurant, Restaurant, Southern Italian restaurant)",32.7224015,-117.1681218,4.518742442563482,1654,1618515764686,45,45,90,301,1173
0x80dbf9fed5bf4b2f:0x4cf3d5b15706bb2c,Olive Garden Italian Restaurant,"Olive Garden Italian Restaurant, 11555 Carmel Mountain Rd, San Diego, CA 92128","List(Italian restaurant, Caterer, Family restaurant, Gluten-free restaurant, Takeout Restaurant, Seafood restaurant, Soup restaurant, Wine bar)",32.978521,-117.082449,4.235013623978202,1468,1618785137043,51,50,152,465,750
0x80d954ae6f3567db:0x40ff164aecf06ffb,Pappalecco,"Pappalecco, 1602 State St, San Diego, CA 92101","List(Italian restaurant, Cafe, Coffee shop, Ice cream shop)",32.722077999999996,-117.1666587,4.664051355206848,1402,1618676897575,12,20,69,225,1076



Writing to Unity Catalog: main.aai510_final_agent.restaurant_metrics
✓ Table created: main.aai510_final_agent.restaurant_metrics

✓ Restaurant metrics table ready for agent tools


---
## 5. Prepare for Vector Search (Review Text)

The current `reviews` table only contains ratings (1-5 stars) without review text. For **Vector Search** and **Complaint/Opportunity Extraction** tools, we need the actual review text to create embeddings.

**Requirements for Vector Search**:
- Review text field for GTE Large embeddings
- Join reviews to filtered San Diego Italian restaurants
- Create Databricks Vector Search index

**Data source**: `review-California.json.gz` contains full review data including text field

In [0]:
# Prepare Review Text for Theme Retrieval

from pyspark.sql import functions as F

# Review text configuration
REVIEW_TEXT_URL = "https://mcauleylab.ucsd.edu:8443/public_datasets/gdrive/googlelocal/review-California.json.gz"
REVIEW_TEXT_PATH = f"/Volumes/{CATALOG}/{SCHEMA}/{VOLUME}/review-California.json.gz"
REVIEW_TEXT_TABLE = f"{CATALOG}.{SCHEMA}.reviews_with_text"

print("=" * 60)
print("Preparing review text for Theme Retrieval...")
print("=" * 60)

print(f"\nReview text source: {REVIEW_TEXT_URL}")
print(f"Review text path: {REVIEW_TEXT_PATH}")
print(f"Review text table: {REVIEW_TEXT_TABLE}")

Preparing review text for Theme Retrieval...

Review text source: https://mcauleylab.ucsd.edu:8443/public_datasets/gdrive/googlelocal/review-California.json.gz
Review text path: /Volumes/main/aai510_final_agent/aai510_final_raw/review-California.json.gz
Review text table: main.aai510_final_agent.reviews_with_text


In [0]:
# Check whether review text table already exists
try:
    reviews_with_text_df = spark.table(REVIEW_TEXT_TABLE)
    
    print(f"✓ Found existing table: {REVIEW_TEXT_TABLE}")
    print(f"✓ Total review-text rows: {reviews_with_text_df.count():,}")
    
    display(reviews_with_text_df.limit(5))
    
    REVIEW_TEXT_READY = True

except Exception:
    print(f"Review text table not found: {REVIEW_TEXT_TABLE}")
    print("Checking whether raw review text file is already staged...")
    
    REVIEW_TEXT_READY = False

✓ Found existing table: main.aai510_final_agent.reviews_with_text
✓ Total review-text rows: 50,113


gmap_id,user_id,rating,time,text,name
0x80dc0185c5e2d4df:0xfbc4e830ef1ca431,114976159291119258205,1,1564203402789,No atmosphere. No fresh ingredients. No variety. The salads ordered has browning lettuce leaves. The fettuccine chicken was completely void of flavor and soggy from the diluted amounts of water. The sandwich ordered was slapped together. Where’s Gordon Ramsey?,Evelynn Herrera
0x80dc0185c5e2d4df:0xfbc4e830ef1ca431,100305772822037390573,4,1501468574646,Traveled from mesa Arizona. was delicious even though we came in a little before closing they still let us sit and eat. Loved the environment and scene. Will definitely come back again.,chris bilagody
0x80dc0185c5e2d4df:0xfbc4e830ef1ca431,110966308572404885088,5,1552792044271,"It is and always has been a wonderful place. Family owned and operated, I have been coming here with my family for 50 years!! They may finally close in September you must check it out if you can. It is a legend to experience at least once. Johnny still plays music at 91. Amazing💝",denise moore
0x80dc0185c5e2d4df:0xfbc4e830ef1ca431,115555920222063468568,4,1556292850077,"Family run for families and music fans. 66 years and closing this September. No more strolling accordion playing, groups singing along, neighborhood restaurant. Not for hip, fast food, quiet, and Yelp critique. It will get low marks. But a last stand for honor, families, hard work, dedication, and community. Proud to have memories.",Suz She
0x80dc0185c5e2d4df:0xfbc4e830ef1ca431,101357089882070703342,5,1557093995582,"There aren't many restaurants that have been around so long and have maintained their original charm as Pernicano's. You will enjoy the family atmosphere, decor that spans decades, and authentic dishes. Parking can be a challenge at busier times, but it's worth the extra effort.",Thom Hiatt


In [0]:
# Stage raw review text file only if needed
if not REVIEW_TEXT_READY:
    
    try:
        dbutils.fs.ls(REVIEW_TEXT_PATH)
        print(f"✓ Already staged: {REVIEW_TEXT_PATH}")
        
    except Exception:
        print("Raw review text file not found.")
        print("Staging raw review text file. This may take several minutes...")
        
        dbutils.fs.cp(REVIEW_TEXT_URL, REVIEW_TEXT_PATH)
        
        print(f"✓ Staged: {REVIEW_TEXT_PATH}")

else:
    print("Skipping raw file staging because review text table already exists.")

Skipping raw file staging because review text table already exists.


In [0]:
# Create review text table only if needed
if not REVIEW_TEXT_READY:
    
    print("=" * 60)
    print("Creating reviews_with_text table...")
    print("=" * 60)
    
    # Read raw review text file
    reviews_with_text_raw_df = spark.read.json(REVIEW_TEXT_PATH)
    
    # Get filtered restaurant IDs from Section 2
    restaurant_ids_df = (
        spark.table(f"{CATALOG}.{SCHEMA}.restaurants")
        .select("gmap_id")
        .distinct()
    )
    
    # Keep only written reviews for selected restaurants
    reviews_with_text_filtered_df = (
        reviews_with_text_raw_df
        .join(restaurant_ids_df, on="gmap_id", how="inner")
        .select(
            "gmap_id",
            "user_id",
            "rating",
            "time",
            "text",
            "name"
        )
        .filter(F.col("text").isNotNull())
        .filter(F.length(F.col("text")) > 10)
    )
    
    # Save filtered review text table
    (
        reviews_with_text_filtered_df
        .write
        .mode("overwrite")
        .option("mergeSchema", "true")
        .saveAsTable(REVIEW_TEXT_TABLE)
    )
    
    reviews_with_text_df = spark.table(REVIEW_TEXT_TABLE)
    
    print(f"✓ Created table: {REVIEW_TEXT_TABLE}")
    print(f"✓ Total review-text rows: {reviews_with_text_df.count():,}")
    
    display(reviews_with_text_df.limit(5))

else:
    print("Skipping table creation because review text table already exists.")

Skipping table creation because review text table already exists.


In [0]:
# Verify review text table
print("=" * 60)
print("Verifying review text table...")
print("=" * 60)

reviews_with_text_df = spark.table(REVIEW_TEXT_TABLE)

print(f"Table: {REVIEW_TEXT_TABLE}")
print(f"Total rows: {reviews_with_text_df.count():,}")

display(
    reviews_with_text_df
    .select("gmap_id", "rating", "text")
    .limit(10)
)

Verifying review text table...
Table: main.aai510_final_agent.reviews_with_text
Total rows: 50,113


gmap_id,rating,text
0x80dc0185c5e2d4df:0xfbc4e830ef1ca431,1,No atmosphere. No fresh ingredients. No variety. The salads ordered has browning lettuce leaves. The fettuccine chicken was completely void of flavor and soggy from the diluted amounts of water. The sandwich ordered was slapped together. Where’s Gordon Ramsey?
0x80dc0185c5e2d4df:0xfbc4e830ef1ca431,4,Traveled from mesa Arizona. was delicious even though we came in a little before closing they still let us sit and eat. Loved the environment and scene. Will definitely come back again.
0x80dc0185c5e2d4df:0xfbc4e830ef1ca431,5,"It is and always has been a wonderful place. Family owned and operated, I have been coming here with my family for 50 years!! They may finally close in September you must check it out if you can. It is a legend to experience at least once. Johnny still plays music at 91. Amazing💝"
0x80dc0185c5e2d4df:0xfbc4e830ef1ca431,4,"Family run for families and music fans. 66 years and closing this September. No more strolling accordion playing, groups singing along, neighborhood restaurant. Not for hip, fast food, quiet, and Yelp critique. It will get low marks. But a last stand for honor, families, hard work, dedication, and community. Proud to have memories."
0x80dc0185c5e2d4df:0xfbc4e830ef1ca431,5,"There aren't many restaurants that have been around so long and have maintained their original charm as Pernicano's. You will enjoy the family atmosphere, decor that spans decades, and authentic dishes. Parking can be a challenge at busier times, but it's worth the extra effort."
0x80dc0185c5e2d4df:0xfbc4e830ef1ca431,5,"Italian food, mmm, super delicious. It was so delicious. Had a friend come into town and asked me to take him here again. The bill was well within reason for the quality. I see why this place has so many great reviews."
0x80dc0185c5e2d4df:0xfbc4e830ef1ca431,4,Fine scene for delicious italian food. Everyone at our table was enjoying everything. Definitely worth going back to. The place has an awesome atmosphere. I see why this place has a lot of positive reviews.
0x80dc0185c5e2d4df:0xfbc4e830ef1ca431,5,"30+ years ago.... I went here semi-regularly with my family, as we lived within walking distance. A few childhood friends' birthday parties were also hosted here. It is a fabulous place and I'm glad to see it still there, at least on Google. My next trip to SD will warrant a visit here with my own kids."
0x80dc0185c5e2d4df:0xfbc4e830ef1ca431,5,I have been going there since 1959 & it is a family atmosphere. Very friendly & Johnny still plays the piano & lets the kids come up with him while playing. The food is real Italian with love. Merry Christmas Johnny & Family
0x80dc0185c5e2d4df:0xfbc4e830ef1ca431,2,"Food was very bland, salad was brown and the only decent thing was garlic bread. Pasta was so bland and watered down. Beer was expensive and small cups were given. Wouldn’t recommend for family dinner. Disappointing."


---
##6. Create Agent Tools

Convert the restaurant data pipeline into reusable tools for the agent. These tools allow the agent to answer both structured questions about ratings and review counts and qualitative questions about customer experiences.

1. **Business Lookup**: Retrieve rating, review count, rating distribution, and restaurant metadata.
2. **Competitor Comparison**: Compare restaurants by rating, review volume, and rating distribution.
3. **Theme Retrieval**: Use vector search over review text to retrieve reviews related to themes like service, food quality, ambience, price, wait time, or complaints.

### 6.1. Configure Source Tables

In [0]:
from pyspark.sql import functions as F

# Source tables
RESTAURANT_METRICS_TABLE = f"{CATALOG}.{SCHEMA}.restaurant_metrics"
RESTAURANTS_TABLE = f"{CATALOG}.{SCHEMA}.restaurants"
REVIEWS_WITH_TEXT_TABLE = f"{CATALOG}.{SCHEMA}.reviews_with_text"
REVIEWS_FOR_VECTOR_TABLE = f"{CATALOG}.{SCHEMA}.reviews_for_vector_search"

print("=" * 60)
print("Configuring agent tool source tables...")
print("=" * 60)

print(f"Restaurant metrics table: {RESTAURANT_METRICS_TABLE}")
print(f"Restaurants table: {RESTAURANTS_TABLE}")
print(f"Review text table: {REVIEWS_WITH_TEXT_TABLE}")
print(f"Vector search source table: {REVIEWS_FOR_VECTOR_TABLE}")

Configuring agent tool source tables...
Restaurant metrics table: main.aai510_final_agent.restaurant_metrics
Restaurants table: main.aai510_final_agent.restaurants
Review text table: main.aai510_final_agent.reviews_with_text
Vector search source table: main.aai510_final_agent.reviews_for_vector_search


### 6.2. Tool 1 - Business Lookup

Objective: Retrieve metrics for a specific restaurant

In [0]:
def business_lookup(restaurant_name: str, limit: int = 5):
    """
    Retrieve rating, review count, rating distribution, and restaurant metadata
    for a specific restaurant.
    """
    
    restaurant_metrics_df = spark.table(RESTAURANT_METRICS_TABLE)
    
    results_df = (
        restaurant_metrics_df
        .filter(F.lower(F.col("business_name")).contains(restaurant_name.lower()))
        .select(
            "gmap_id",
            "business_name",
            "address",
            "avg_rating",
            "review_count",
            "rating_1_count",
            "rating_2_count",
            "rating_3_count",
            "rating_4_count",
            "rating_5_count"
        )
        .orderBy(F.desc("review_count"))
        .limit(limit)
    )
    
    results = results_df.toPandas().to_dict(orient="records")
    
    if len(results) == 0:
        return {
            "status": "not_found",
            "message": f"No restaurant found matching '{restaurant_name}'."
        }
    
    return {
        "status": "success",
        "results": results
    }

### 6.3. Tool 2 - Competitor Comparison

Objective: Compare restaurant performance to another restaurant

In [0]:
def competitor_comparison(restaurant_1: str, restaurant_2: str, limit_per_name: int = 3):
    """
    Compare two specific restaurants by rating, review volume, and rating distribution.
    The user provides both restaurant names.
    """
    
    restaurant_metrics_df = spark.table(RESTAURANT_METRICS_TABLE)
    
    restaurant_names = [restaurant_1, restaurant_2]
    all_results = []
    
    for restaurant_name in restaurant_names:
        matches_df = (
            restaurant_metrics_df
            .filter(F.lower(F.col("business_name")).contains(restaurant_name.lower()))
            .select(
                "gmap_id",
                "business_name",
                "address",
                "avg_rating",
                "review_count",
                "rating_1_count",
                "rating_2_count",
                "rating_3_count",
                "rating_4_count",
                "rating_5_count"
            )
            .orderBy(F.desc("review_count"))
            .limit(limit_per_name)
        )
        
        matches = matches_df.toPandas().to_dict(orient="records")
        
        for match in matches:
            match["search_name"] = restaurant_name
        
        all_results.extend(matches)
    
    if len(all_results) == 0:
        return {
            "status": "not_found",
            "message": f"No matching restaurants found for '{restaurant_1}' or '{restaurant_2}'."
        }
    
    return {
        "status": "success",
        "restaurant_1": restaurant_1,
        "restaurant_2": restaurant_2,
        "results": all_results
    }

### 6.4. Prepare Review Text Table for Vector Search

Vector Search needs a stable primary key. This creates a new table with a `review_id`

To keep vector search efficient while making the agent more generalizable, this section indexes all available review text for the two selected comparison restaurants and a random sample of 20 additional San Diego Italian restaurants. This allows the agent to answer the primary comparison question while still supporting theme retrieval for other restaurants in the dataset.

In [0]:
# Create a vector-search-ready review text table with a stable primary key
# Include the two selected restaurants plus a random sample of other restaurants

SELECTED_RESTAURANTS = ["cesarina", "parma"]
NUM_RANDOM_RESTAURANTS = 20
MAX_REVIEWS_PER_RANDOM_RESTAURANT = 50

reviews_with_text_df = spark.table(REVIEWS_WITH_TEXT_TABLE)
restaurants_df = spark.table(RESTAURANTS_TABLE)

reviews_joined_df = (
    reviews_with_text_df
    .join(
        restaurants_df.select("gmap_id", "business_name", "address"),
        on="gmap_id",
        how="inner"
    )
    .filter(F.col("text").isNotNull())
    .filter(F.length(F.col("text")) > 10)
)

# Keep all reviews for selected restaurants
selected_reviews_df = (
    reviews_joined_df
    .filter(
        (F.lower(F.col("business_name")).contains("cesarina")) |
        (F.lower(F.col("business_name")).contains("parma"))
    )
)

# Randomly choose additional restaurants, excluding selected restaurants
random_restaurant_ids_df = (
    reviews_joined_df
    .filter(
        ~(
            (F.lower(F.col("business_name")).contains("cesarina")) |
            (F.lower(F.col("business_name")).contains("parma"))
        )
    )
    .select("gmap_id", "business_name")
    .distinct()
    .orderBy(F.rand(seed=42))
    .limit(NUM_RANDOM_RESTAURANTS)
)

# For those random restaurants, take up to N reviews per restaurant
random_reviews_base_df = (
    reviews_joined_df
    .join(
        random_restaurant_ids_df.select("gmap_id"),
        on="gmap_id",
        how="inner"
    )
)

window_by_restaurant = Window.partitionBy("gmap_id").orderBy(F.rand(seed=42))

random_reviews_df = (
    random_reviews_base_df
    .withColumn("random_review_rank", F.row_number().over(window_by_restaurant))
    .filter(F.col("random_review_rank") <= MAX_REVIEWS_PER_RANDOM_RESTAURANT)
    .drop("random_review_rank")
)

# Combine selected restaurants and random sample
reviews_for_vector_df = (
    selected_reviews_df
    .unionByName(random_reviews_df)
    .select(
        F.sha2(
            F.concat_ws(
                "||",
                F.col("gmap_id"),
                F.coalesce(F.col("user_id"), F.lit("unknown_user")),
                F.coalesce(F.col("time").cast("string"), F.lit("unknown_time")),
                F.coalesce(F.col("text"), F.lit(""))
            ),
            256
        ).alias("review_id"),
        "gmap_id",
        "business_name",
        "address",
        "rating",
        "time",
        "text"
    )
    .dropDuplicates(["review_id"])
)

(
    reviews_for_vector_df
    .write
    .format("delta")
    .mode("overwrite")
    .option("mergeSchema", "true")
    .option("delta.enableChangeDataFeed", "true")
    .saveAsTable(REVIEWS_FOR_VECTOR_TABLE)
)

# Change Data Feed is required for Delta Sync Vector Search indexes
spark.sql(f"""
ALTER TABLE {REVIEWS_FOR_VECTOR_TABLE}
SET TBLPROPERTIES (delta.enableChangeDataFeed = true)
""")

print(f"Created vector-search source table: {REVIEWS_FOR_VECTOR_TABLE}")
print(f"Total rows: {spark.table(REVIEWS_FOR_VECTOR_TABLE).count():,}")

display(
    spark.table(REVIEWS_FOR_VECTOR_TABLE)
    .groupBy("business_name")
    .agg(F.count("*").alias("review_text_count"))
    .orderBy(F.desc("review_text_count"))
)

Created vector-search source table: main.aai510_final_agent.reviews_for_vector_search
Total rows: 1,503


business_name,review_text_count
Parma Cucina Italiana,349
Cesarina,320
Round Table Pizza,100
RoVino The Foodery,50
Ulivo Restaurant,50
Pernicano's Ristorante,50
Villa Capri,50
Mia Trattoria,50
Piacere Mio,50
Il Postino Restaurant,50


### 6.5 Initialize Vector Search Client

In [0]:
from databricks.vector_search.client import VectorSearchClient

vsc = VectorSearchClient()

print("Vector Search client initialized.")

[NOTICE] Using a notebook authentication token. Recommended for development only. For improved performance, please use Service Principal based authentication. To disable this message, pass disable_notice=True.
Vector Search client initialized.


### 6.6. Configure Vector Search Endpoint and Index

In [0]:
# Vector Search configuration
VECTOR_SEARCH_ENDPOINT_NAME = "restaurant_reviews_vs_endpoint"
VECTOR_SEARCH_INDEX_NAME = f"{CATALOG}.{SCHEMA}.restaurant_review_text_index"

# Databricks embedding model endpoint
# If this endpoint is not available in your workspace, replace it with the embedding endpoint your course/workspace provides.
EMBEDDING_MODEL_ENDPOINT_NAME = "databricks-gte-large-en"

print("=" * 60)
print("Configuring Vector Search index...")
print("=" * 60)

print(f"Endpoint: {VECTOR_SEARCH_ENDPOINT_NAME}")
print(f"Index: {VECTOR_SEARCH_INDEX_NAME}")
print(f"Source table: {REVIEWS_FOR_VECTOR_TABLE}")
print(f"Embedding endpoint: {EMBEDDING_MODEL_ENDPOINT_NAME}")

Configuring Vector Search index...
Endpoint: restaurant_reviews_vs_endpoint
Index: main.aai510_final_agent.restaurant_review_text_index
Source table: main.aai510_final_agent.reviews_for_vector_search
Embedding endpoint: databricks-gte-large-en


### 6.7. Create or Connect to Vector Search Endpoint

Create an endpoint/index if needed, otherwise connect to the existing one

In [0]:
# Create Vector Search endpoint if needed

try:
    vsc.get_endpoint(VECTOR_SEARCH_ENDPOINT_NAME)
    print(f"✓ Found existing endpoint: {VECTOR_SEARCH_ENDPOINT_NAME}")

except Exception:
    print(f"Creating endpoint: {VECTOR_SEARCH_ENDPOINT_NAME}")
    vsc.create_endpoint(
        name=VECTOR_SEARCH_ENDPOINT_NAME,
        endpoint_type="STANDARD"
    )
    vsc.wait_for_endpoint(VECTOR_SEARCH_ENDPOINT_NAME)
    print(f"✓ Created endpoint: {VECTOR_SEARCH_ENDPOINT_NAME}")

Creating endpoint: restaurant_reviews_vs_endpoint
✓ Created endpoint: restaurant_reviews_vs_endpoint


### 6.8. Create or Connect to Vector Search Index

In [0]:
# Delete offline / partial Vector Search index

try:
    print(f"Deleting index: {VECTOR_SEARCH_INDEX_NAME}")
    
    vsc.delete_index(
        endpoint_name=VECTOR_SEARCH_ENDPOINT_NAME,
        index_name=VECTOR_SEARCH_INDEX_NAME
    )
    
    print(f"Deleted index: {VECTOR_SEARCH_INDEX_NAME}")

except Exception as e:
    print("Index delete failed or index was already gone.")
    print(e)

Deleting index: main.aai510_final_agent.restaurant_review_text_index
Deleted index: main.aai510_final_agent.restaurant_review_text_index


In [0]:
# Create Delta Sync index if needed

try:
    review_index = vsc.get_index(
        endpoint_name=VECTOR_SEARCH_ENDPOINT_NAME,
        index_name=VECTOR_SEARCH_INDEX_NAME
    )
    print(f"✓ Found existing index: {VECTOR_SEARCH_INDEX_NAME}")

except Exception:
    print(f"Creating index: {VECTOR_SEARCH_INDEX_NAME}")
    
    review_index = vsc.create_delta_sync_index(
        endpoint_name=VECTOR_SEARCH_ENDPOINT_NAME,
        index_name=VECTOR_SEARCH_INDEX_NAME,
        source_table_name=REVIEWS_FOR_VECTOR_TABLE,
        pipeline_type="TRIGGERED",
        primary_key="review_id",
        embedding_source_column="text",
        embedding_model_endpoint_name=EMBEDDING_MODEL_ENDPOINT_NAME,
        columns_to_sync=[
            "review_id",
            "gmap_id",
            "business_name",
            "address",
            "rating",
            "time",
            "text"
        ]
    )
    
    review_index.wait_until_ready()
    print(f"✓ Created index: {VECTOR_SEARCH_INDEX_NAME}")

Creating index: main.aai510_final_agent.restaurant_review_text_index
✓ Created index: main.aai510_final_agent.restaurant_review_text_index


### 6.9. Sync Vector Search Index

In [0]:
# Sync the index after creating or updating the source table

review_index = vsc.get_index(
    endpoint_name=VECTOR_SEARCH_ENDPOINT_NAME,
    index_name=VECTOR_SEARCH_INDEX_NAME
)

review_index.sync()
review_index.wait_until_ready()

print("✓ Vector Search index is synced and ready.")

✓ Vector Search index is synced and ready.


### 6.10. Helper Function to Parse Vector Search Results

In [0]:
def parse_vector_search_results(results):
    """
    Convert Databricks Vector Search response into a list of dictionaries.
    """
    
    columns = [col["name"] for col in results["manifest"]["columns"]]
    rows = results["result"]["data_array"]
    
    parsed_results = []
    
    for row in rows:
        parsed_results.append(dict(zip(columns, row)))
    
    return parsed_results

### 6.11. Tool 3 - Theme Retrieval

In [0]:
def theme_retrieval(restaurant_name: str, theme_query: str, num_results: int = 10):
    """
    Retrieve reviews related to a user-provided theme using Databricks Vector Search.
    This uses semantic similarity search over review text and does not rely on
    hard-coded keywords.
    """
    
    review_index = vsc.get_index(
        endpoint_name=VECTOR_SEARCH_ENDPOINT_NAME,
        index_name=VECTOR_SEARCH_INDEX_NAME
    )
    
    # Pull more than needed because restaurant filtering happens after retrieval.
    raw_num_results = max(num_results * 5, 25)
    
    results = review_index.similarity_search(
        query_text=theme_query,
        columns=[
            "review_id",
            "business_name",
            "address",
            "rating",
            "time",
            "text"
        ],
        num_results=raw_num_results
    )
    
    parsed_results = parse_vector_search_results(results)
    
    filtered_results = [
        row for row in parsed_results
        if restaurant_name.lower() in row.get("business_name", "").lower()
    ]
    
    filtered_results = filtered_results[:num_results]
    
    if len(filtered_results) == 0:
        return {
            "status": "not_found",
            "message": (
                f"No vector search results found for restaurant '{restaurant_name}' "
                f"and theme query '{theme_query}'."
            ),
            "restaurant_name": restaurant_name,
            "theme_query": theme_query,
            "results": []
        }
    
    return {
        "status": "success",
        "restaurant_name": restaurant_name,
        "theme_query": theme_query,
        "results": filtered_results
    }

### 6.12 Validate All Three Tools

In [0]:
print("=" * 60)
print("Validating agent tools...")
print("=" * 60)

print("\nTool 1: Business Lookup")
business_lookup_result = business_lookup("Cesarina")

if business_lookup_result["status"] == "success":
    display(business_lookup_result["results"])
else:
    print(business_lookup_result["message"])


print("\nTool 2: Competitor Comparison")
comparison_result = competitor_comparison("Cesarina", "Parma Cucina Italiana")

if comparison_result["status"] == "success":
    display(comparison_result["results"])
else:
    print(comparison_result["message"])


print("\nTool 3: Theme Retrieval")
try:
    theme_result = theme_retrieval(
        "Cesarina",
        "reviews about pasta quality and authentic Italian food",
        num_results=5
    )
    
    if theme_result["status"] == "success":
        display(theme_result["results"])
    else:
        print(theme_result["message"])

except Exception as e:
    print("Theme Retrieval failed.")
    print(e)

Validating agent tools...

Tool 1: Business Lookup


address,avg_rating,business_name,gmap_id,rating_1_count,rating_2_count,rating_3_count,rating_4_count,rating_5_count,review_count
"Cesarina, 4161 Voltaire St, San Diego, CA 92107, United States",4.840304182509506,Cesarina,0x80deaa30b08c1367:0x807034f5191ec879,2,2,11,48,463,526



Tool 2: Competitor Comparison


address,avg_rating,business_name,gmap_id,rating_1_count,rating_2_count,rating_3_count,rating_4_count,rating_5_count,review_count,search_name
"Cesarina, 4161 Voltaire St, San Diego, CA 92107, United States",4.840304182509506,Cesarina,0x80deaa30b08c1367:0x807034f5191ec879,2,2,11,48,463,526,Cesarina
"Parma Cucina Italiana, 3850 Fifth Ave, San Diego, CA 92103",4.670289855072464,Parma Cucina Italiana,0x80d954da94044cfb:0x9830f72553b8350c,7,7,24,85,429,552,Parma Cucina Italiana



Tool 3: Theme Retrieval
[NOTICE] Using a notebook authentication token. Recommended for development only. For improved performance, please use Service Principal based authentication. To disable this message, pass disable_notice=True.


address,business_name,rating,review_id,score,text,time
"Cesarina, 4161 Voltaire St, San Diego, CA 92107, United States",Cesarina,5.0,c8bcf70b7a4bf731ac20a3230af624139cd169b14f6b4e63ec1d93b93d1cac21,0.683217485245215,Amazing fresh pasta,1.578022539062E12
"Cesarina, 4161 Voltaire St, San Diego, CA 92107, United States",Cesarina,5.0,af652a30074e57e2e35b79a5286239ace603176d61cab25dffcb0413c49b565f,0.6708889848396832,"Excellent homemade pasta, highly recommended",1.560981598778E12
"Cesarina, 4161 Voltaire St, San Diego, CA 92107, United States",Cesarina,5.0,c32b474c7073b02a6db1590dae2428250ca30615cfa91453e6fb87a5fe9d11c1,0.666536010261426,"Seriously good, real home made pastas, must try gnocchi, pappardelle, rigatoni. Pick your own sauce and of course owners are so acomodating. Great experience thank you",1.559279660709E12
"Cesarina, 4161 Voltaire St, San Diego, CA 92107, United States",Cesarina,5.0,759b96259509e0be622c3e5ae4ebedebba8bceb5ce400a4de1538620ed05880c,0.6656511112651599,What a wonderful discovery of excellent of fresh italian pasta with excellent service! Yum!,1.564346874129E12
"Cesarina, 4161 Voltaire St, San Diego, CA 92107, United States",Cesarina,5.0,dde684d7e241ce7103b7a170f3341734a0bad48ddd8c4caa38d554db23a93c67,0.6565337575663645,"Outstanding Italian restaurant with great ambiance. The food in general, and pasta in particular, deserve six stars!!! Everything is made fresh and taste very authentic, like as if it was in the heart of Italy. Highly recommended.",1.550029505853E12


---
## 7. Create Restaurant Comparison Agent

This section connects the reusable restaurant tools to a tool-calling agent. The agent uses an LLM to interpret the user’s question, choose the appropriate tool, call the tool, and synthesize a final answer using the tool results.

### 7.1. Configure LLM Endpoint

Baseline: Llama 3.3 70B
Comparirson: GPT OSS 20b (cheaper model)

In [0]:
# LLM endpoints used for the restaurant comparison agent

LLAMA_3_3_70B_ENDPOINT = "databricks-meta-llama-3-3-70b-instruct"
GPT_OSS_20B_ENDPOINT = "databricks-gpt-oss-20b"

AVAILABLE_LLMS = [
    LLAMA_3_3_70B_ENDPOINT,
    GPT_OSS_20B_ENDPOINT
]

# Choose the default LLM for the first agent run
CHOSEN_LLM_ENDPOINT = LLAMA_3_3_70B_ENDPOINT

print("=" * 60)
print("Configuring LLM endpoints...")
print("=" * 60)

print("Available LLM endpoints:")
for endpoint in AVAILABLE_LLMS:
    print(f"- {endpoint}")

print(f"\nDefault LLM endpoint: {CHOSEN_LLM_ENDPOINT}")

Configuring LLM endpoints...
Available LLM endpoints:
- databricks-meta-llama-3-3-70b-instruct
- databricks-gpt-oss-20b

Default LLM endpoint: databricks-meta-llama-3-3-70b-instruct


### 7.2. Define Tool Metadata

In [0]:
import json
from dataclasses import dataclass
from typing import Callable

@dataclass
class ToolInfo:
    name: str
    spec: dict
    exec_fn: Callable

### 7.3. Define Tool Specifications

In [0]:
# Tool specifications in OpenAI-compatible function-calling format

business_lookup_spec = {
    "type": "function",
    "function": {
        "name": "business_lookup",
        "description": (
            "Retrieve rating, review count, rating distribution, and restaurant metadata "
            "for a specific restaurant."
        ),
        "parameters": {
            "type": "object",
            "properties": {
                "restaurant_name": {
                    "type": "string",
                    "description": "Name or partial name of the restaurant to look up."
                },
                "limit": {
                    "type": "integer",
                    "description": "Maximum number of matching restaurants to return."
                }
            },
            "required": ["restaurant_name"]
        }
    }
}


competitor_comparison_spec = {
    "type": "function",
    "function": {
        "name": "competitor_comparison",
        "description": (
            "Compare two specific restaurants by rating, review volume, and rating distribution."
        ),
        "parameters": {
            "type": "object",
            "properties": {
                "restaurant_1": {
                    "type": "string",
                    "description": "Name or partial name of the first restaurant."
                },
                "restaurant_2": {
                    "type": "string",
                    "description": "Name or partial name of the second restaurant."
                },
                "limit_per_name": {
                    "type": "integer",
                    "description": "Maximum number of matches to return per restaurant name."
                }
            },
            "required": ["restaurant_1", "restaurant_2"]
        }
    }
}


theme_retrieval_spec = {
    "type": "function",
    "function": {
        "name": "theme_retrieval",
        "description": (
            "Retrieve reviews related to a user-provided theme using semantic Vector Search "
            "over restaurant review text. Use this for qualitative questions about customer experiences, "
            "including service, food quality, ambiance, price, wait time, complaints, and special occasions."
        ),
        "parameters": {
            "type": "object",
            "properties": {
                "restaurant_name": {
                    "type": "string",
                    "description": "Name or partial name of the restaurant."
                },
                "theme_query": {
                    "type": "string",
                    "description": (
                        "Natural-language theme to search for in review text. "
                        "For example: 'service and staff friendliness', "
                        "'pasta quality and authentic Italian food', or "
                        "'ambiance for date night'."
                    )
                },
                "num_results": {
                    "type": "integer",
                    "description": "Number of relevant review excerpts to retrieve."
                }
            },
            "required": ["restaurant_name", "theme_query"]
        }
    }
}

### 7.4. Register Tools

In [0]:
TOOL_INFOS = [
    ToolInfo(
        name="business_lookup",
        spec=business_lookup_spec,
        exec_fn=business_lookup
    ),
    ToolInfo(
        name="competitor_comparison",
        spec=competitor_comparison_spec,
        exec_fn=competitor_comparison
    ),
    ToolInfo(
        name="theme_retrieval",
        spec=theme_retrieval_spec,
        exec_fn=theme_retrieval
    )
]

print("=" * 60)
print("Registering agent tools...")
print("=" * 60)

print(f"Registered {len(TOOL_INFOS)} tools:")
for tool in TOOL_INFOS:
    print(f"- {tool.name}")

Registering agent tools...
Registered 3 tools:
- business_lookup
- competitor_comparison
- theme_retrieval


### 7.5. Create System Prompt

In [0]:
SYSTEM_PROMPT = """
You are a restaurant comparison assistant focused on San Diego Italian restaurants.

Your scope:
- Answer questions about San Diego Italian restaurants using the available restaurant metrics and review-text tools.
- Help users compare restaurants, understand ratings, review counts, rating distributions, and customer review themes.
- Provide recommendations only when they can be grounded in the available tool outputs.

Out-of-scope requests:
- Do not answer questions unrelated to San Diego Italian restaurants.
- Do not provide general homework help, coding help, medical advice, legal advice, financial advice, travel planning, politics, sports, entertainment, or unrelated factual information.
- Do not invent restaurant data, restaurant names, metrics, ratings, review counts, review text, or business recommendations.

If the user asks an out-of-scope question, politely reject it and briefly explain that you can only help with San Diego Italian restaurant comparison questions using the available data.

You have access to three tools:
1. business_lookup: retrieves restaurant-level metrics such as average rating, review count, and rating distribution.
2. competitor_comparison: compares two restaurants using rating, review volume, and rating distribution.
3. theme_retrieval: retrieves semantically relevant review excerpts for qualitative customer-experience themes.

Tool-use rules:
- For structured questions about one restaurant, use business_lookup.
- For structured comparison questions about two restaurants, use competitor_comparison.
- For qualitative questions about customer experiences, use theme_retrieval.
- For broad comparison, recommendation, or “what does one restaurant do better” questions, use both competitor_comparison and theme_retrieval. The structured comparison shows rating and review-count differences, while theme_retrieval provides specific evidence about what customers mention, such as service, food quality, ambiance, authenticity, value, wait time, and special occasions.
- If a tool returns no results, say that the data was not found.

Use the restaurant names provided by the user. Do not assume the user only cares about a fixed pair of restaurants.
Be concise, but include enough evidence from tool outputs to justify the answer.
"""

### 7.6. Create Tool-Calling Agent Class with MLflow Tracing

In [0]:
import mlflow
from databricks.sdk import WorkspaceClient

# Enable MLflow tracing for OpenAI-compatible chat calls
mlflow.openai.autolog()

print("MLflow OpenAI autologging enabled.")

MLflow OpenAI autologging enabled.


In [0]:
class RestaurantToolCallingAgent:
    def __init__(self, llm_endpoint: str, tools: list[ToolInfo]):
        self.llm_endpoint = llm_endpoint
        self.tools = {tool.name: tool for tool in tools}
        self.workspace_client = WorkspaceClient()
        self.client = self.workspace_client.serving_endpoints.get_open_ai_client()
    
    def get_tool_specs(self):
        return [tool.spec for tool in self.tools.values()]
    
    def execute_tool(self, tool_name: str, arguments: dict):
        if tool_name not in self.tools:
            return {
                "status": "error",
                "message": f"Unknown tool: {tool_name}"
            }
        
        tool = self.tools[tool_name]
        return tool.exec_fn(**arguments)
    
    def predict(self, user_question: str, max_tool_rounds: int = 5, verbose: bool = True):
        messages = [
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": user_question}
        ]
        
        for _ in range(max_tool_rounds):
            response = self.client.chat.completions.create(
                model=self.llm_endpoint,
                messages=messages,
                tools=self.get_tool_specs(),
                tool_choice="auto"
            )
            
            assistant_message = response.choices[0].message
            messages.append(assistant_message.model_dump(exclude_none=True))
            
            # If the model does not call a tool, return the final response
            if not assistant_message.tool_calls:
                return assistant_message.content
            
            # Execute tool calls
            for tool_call in assistant_message.tool_calls:
                tool_name = tool_call.function.name
                arguments = json.loads(tool_call.function.arguments or "{}")
                
                if verbose:
                    print(f"Calling tool: {tool_name}")
                    print(f"Arguments: {arguments}")
                
                tool_result = self.execute_tool(tool_name, arguments)
                
                messages.append({
                    "role": "tool",
                    "tool_call_id": tool_call.id,
                    "content": json.dumps(tool_result, default=str)
                })
        
        return "The agent reached the maximum number of tool calls before producing a final answer."

### 7.7. Create Agents for Both LLMs

In [0]:
AGENT_LLAMA_3_3_70B = RestaurantToolCallingAgent(
    llm_endpoint=LLAMA_3_3_70B_ENDPOINT,
    tools=TOOL_INFOS
)

AGENT_GPT_OSS_20B = RestaurantToolCallingAgent(
    llm_endpoint=GPT_OSS_20B_ENDPOINT,
    tools=TOOL_INFOS
)

# Default agent
AGENT = AGENT_LLAMA_3_3_70B

print("=" * 60)
print("Restaurant comparison agents created.")
print("=" * 60)

print(f"Default agent: {LLAMA_3_3_70B_ENDPOINT}")
print(f"Comparison agent: {GPT_OSS_20B_ENDPOINT}")

Restaurant comparison agents created.
Default agent: databricks-meta-llama-3-3-70b-instruct
Comparison agent: databricks-gpt-oss-20b


### 7.8. Compare Both LLMs on the Same Prompts

#### Prompt 1

In [0]:
test_question = (
    "How many reviews does Parma Cucina Italiana have, and what is its average rating?"
)

print("=" * 80)
print("Llama 3.3 70B")
print("=" * 80)

llama_answer = AGENT_LLAMA_3_3_70B.predict(test_question)
print(llama_answer)


print("\n" + "=" * 80)
print("GPT OSS 20B")
print("=" * 80)

gpt_oss_answer = AGENT_GPT_OSS_20B.predict(test_question)
print(gpt_oss_answer)

Llama 3.3 70B
Calling tool: business_lookup
Arguments: {'restaurant_name': 'Parma Cucina Italiana', 'limit': 1}
Parma Cucina Italiana has an average rating of 4.67 and 552 reviews.

GPT OSS 20B
Calling tool: business_lookup
Arguments: {'restaurant_name': 'Parma Cucina Italiana', 'limit': 1}
**Parma Cucina Italiana**

| Metric | Value |
|--------|-------|
| **Avg. rating** | 4.67 ⭐ |
| **Total reviews** | 552 |

The data comes from a direct `business_lookup` of “Parma Cucina Italiana.”


[Trace(trace_id=tr-0302c15d7627e9d3f0185d1a7a55beb0), Trace(trace_id=tr-ec76961e91739038801b392290aea05b), Trace(trace_id=tr-aeee5a5bda7b371fc482a9647ec7421c), Trace(trace_id=tr-b6cf203c0f285a0974730053423a104b)]

#### Prompt 2

In [0]:
test_question = (
    "Compare Parma Cucina Italiana and Cesarina. What does Cesarina appear to do better?"
)

print("=" * 80)
print("Llama 3.3 70B")
print("=" * 80)

llama_answer = AGENT_LLAMA_3_3_70B.predict(test_question)
print(llama_answer)


print("\n" + "=" * 80)
print("GPT OSS 20B")
print("=" * 80)

gpt_oss_answer = AGENT_GPT_OSS_20B.predict(test_question)
print(gpt_oss_answer)

Llama 3.3 70B
Calling tool: competitor_comparison
Arguments: {'restaurant_1': 'Parma Cucina Italiana', 'restaurant_2': 'Cesarina', 'limit_per_name': 1}
Calling tool: theme_retrieval
Arguments: {'restaurant_name': 'Cesarina', 'theme_query': 'food quality and authentic Italian food', 'num_results': 5}
[NOTICE] Using a notebook authentication token. Recommended for development only. For improved performance, please use Service Principal based authentication. To disable this message, pass disable_notice=True.
Based on the comparison of Parma Cucina Italiana and Cesarina, Cesarina appears to have a higher average rating (4.84 vs 4.67) and more 5-star reviews (463 vs 429). The theme retrieval results for Cesarina also suggest that it excels in terms of food quality and authentic Italian cuisine, with reviewers praising its "authentic Italian" dishes, "terrific" and "genuine" Italian food, and "phenomenal" authentic Italian cuisine. Overall, Cesarina seems to have an edge over Parma Cucina It

[Trace(trace_id=tr-6e3fa3ce93dea10781b991cc8170fb68), Trace(trace_id=tr-c8a6474e3d9f2f5d1df99fd411f91a0f), Trace(trace_id=tr-c738ddb1dbbf48549393fefd4afab872), Trace(trace_id=tr-d3c676f4d0b23338ab7cd4737be678a9), Trace(trace_id=tr-0638f6df95f2f3f39bc333c93409216f), Trace(trace_id=tr-a6c719207a98d8661dd996e01995b7b3), Trace(trace_id=tr-b442e3e7334a4b1eba2e3b13ca42b225), Trace(trace_id=tr-7a566756a7ead0e1bd567f0f01f45aec)]

#### Prompt 3

In [0]:
test_question = (
    "What are customers saying about the pasta quality and Italian authenticity at Parma Cucina Italiana?"
)

print("=" * 80)
print("Llama 3.3 70B")
print("=" * 80)

llama_answer = AGENT_LLAMA_3_3_70B.predict(test_question)
print(llama_answer)


print("\n" + "=" * 80)
print("GPT OSS 20B")
print("=" * 80)

gpt_oss_answer = AGENT_GPT_OSS_20B.predict(test_question)
print(gpt_oss_answer)

Llama 3.3 70B
Calling tool: theme_retrieval
Arguments: {'restaurant_name': 'Parma Cucina Italiana', 'theme_query': 'pasta quality and Italian authenticity', 'num_results': 5}
[NOTICE] Using a notebook authentication token. Recommended for development only. For improved performance, please use Service Principal based authentication. To disable this message, pass disable_notice=True.
Based on the review excerpts, customers are saying that Parma Cucina Italiana has "authentic Italian" cuisine, with one reviewer mentioning it is "Very good and authentic Italian". Another reviewer simply states that it serves "Great Italian food". The reviews suggest that the pasta quality and Italian authenticity at Parma Cucina Italiana are well-regarded by customers, with ratings of 4.0 and 5.0 out of 5.0. Overall, the feedback indicates a positive perception of the restaurant's Italian cuisine.

GPT OSS 20B
Calling tool: theme_retrieval
Arguments: {'restaurant_name': 'Parma Cucina Italiana', 'theme_quer

[Trace(trace_id=tr-d93641c30349722319fe3e6e56c47ef6), Trace(trace_id=tr-7d48989621886b81cf7ddb5ad4267e6a), Trace(trace_id=tr-8443e100674856fbf5cc432b6a1d41f6), Trace(trace_id=tr-5a0649752966a6be463a2c83642ec73a)]

#### Prompt 4

In [0]:
test_question = (
    "What is the best recipe for homemade lasagna?"
)

print("=" * 80)
print("Llama 3.3 70B")
print("=" * 80)

llama_answer = AGENT_LLAMA_3_3_70B.predict(test_question)
print(llama_answer)


print("\n" + "=" * 80)
print("GPT OSS 20B")
print("=" * 80)

gpt_oss_answer = AGENT_GPT_OSS_20B.predict(test_question)
print(gpt_oss_answer)

Llama 3.3 70B
I can only help with San Diego Italian restaurant comparison questions using the available data. I don't have information about recipes or cooking instructions. If you have a question about a specific San Diego Italian restaurant, such as their ratings, review counts, or customer experiences, I'd be happy to try and assist you.

GPT OSS 20B
[{'type': 'reasoning', 'summary': [{'type': 'summary_text', 'text': 'The user is asking a general recipe. That is outside scope. We must refuse and explain we only help with San Diego Italian restaurant comparison.'}]}, {'type': 'text', 'text': 'I’m sorry, but I can’t help with that. I’m only able to answer questions about San Diego Italian restaurants using the available data.'}]


[Trace(trace_id=tr-621b347a0e6af3b2ce9e7808c279471f), Trace(trace_id=tr-8f9913e60449faf8db8b9b0c47f6726d)]

#### Prompt 5

In [0]:
test_question = (
    "Can you summarize reviews for a dentist in New York?"
)

print("=" * 80)
print("Llama 3.3 70B")
print("=" * 80)

llama_answer = AGENT_LLAMA_3_3_70B.predict(test_question)
print(llama_answer)


print("\n" + "=" * 80)
print("GPT OSS 20B")
print("=" * 80)

gpt_oss_answer = AGENT_GPT_OSS_20B.predict(test_question)
print(gpt_oss_answer)

Llama 3.3 70B
I can only help with San Diego Italian restaurant comparison questions using the available data. Unfortunately, I don't have information about dentists in New York, as it's outside my scope. If you have any questions about San Diego Italian restaurants, I'd be happy to try and assist you.

GPT OSS 20B
[{'type': 'reasoning', 'summary': [{'type': 'summary_text', 'text': 'User asks about dentist in New York, which is out of scope. Must politely reject.'}]}, {'type': 'text', 'text': 'I’m sorry, but I can only help with questions about San\u202fDiego Italian restaurants. If you have any questions in that area, feel free to ask!'}]


[Trace(trace_id=tr-136a6b3bce24cf7b5579341d9114f6c0), Trace(trace_id=tr-8f7fad67b4d7bc1fd9c7ecbab9c811e3)]

## 8. Evaluation Agent with LLM Judges

This section evaluates the restaurant comparison agent using LLM-as-a-judge scoring. The evaluation checks whether the agent answers the user’s question, stays grounded in tool outputs, uses the appropriate tools, and gracefully rejects out-of-scope questions. The evaluation is run through MLflow so that traces and judge scores can be reviewed in the Databricks Experiments UI.

### 8.1. Set MLflow Experiment

In [0]:
import mlflow

# Set MLflow experiment for final project traces and evaluation
current_user = spark.sql("SELECT current_user()").first()[0]
EXPERIMENT_NAME = f"/Users/{current_user}/aai510_restaurant_agent_final"

mlflow.set_experiment(EXPERIMENT_NAME)

print("=" * 60)
print("MLflow experiment configured...")
print("=" * 60)
print(f"Experiment: {EXPERIMENT_NAME}")

2026/06/17 03:46:42 INFO mlflow.tracking.fluent: Experiment with name '/Users/erika3gallegos@gmail.com/aai510_restaurant_agent_final' does not exist. Creating a new experiment.


MLflow experiment configured...
Experiment: /Users/erika3gallegos@gmail.com/aai510_restaurant_agent_final


If you are using MLflow Tracing, consider storing your traces in Unity Catalog for unlimited storage (no 100,000 trace limit), fine-grained access controls, and queryability from notebooks, SQL, and dashboards. Learn more: https://docs.databricks.com/aws/en/mlflow3/genai/tracing/trace-unity-catalog


### 8.2. Create Evaluation Questions

In [0]:
# Evaluation questions for LLM judge
# These are the same types of prompts tested in Section 7

EVALUATION_QUESTIONS = [
    {
        "question": "How many reviews does Parma Cucina Italiana have, and what is its average rating?",
        "expected_tools": ["business_lookup"]
    },
    {
        "question": "Compare Parma Cucina Italiana and Cesarina. What does Cesarina appear to do better?",
        "expected_tools": ["competitor_comparison"]
    },
    {
        "question": (
            "What are customers saying about the service and staff friendliness at Parma Cucina Italiana?"
        ),
        "expected_tools": ["theme_retrieval"]
    },
    {
        "question": "What is the best recipe for homemade lasagna?",
        "expected_tools": []
    },
    {
        "question": "Can you summarize reviews for a dentist in New York?",
        "expected_tools": []
    }
]

print("=" * 60)
print("Evaluation questions created...")
print("=" * 60)
print(f"Total evaluation questions: {len(EVALUATION_QUESTIONS)}")

display(EVALUATION_QUESTIONS)

Evaluation questions created...
Total evaluation questions: 5


expected_tools,question
List(business_lookup),"How many reviews does Parma Cucina Italiana have, and what is its average rating?"
List(competitor_comparison),Compare Parma Cucina Italiana and Cesarina. What does Cesarina appear to do better?
List(theme_retrieval),What are customers saying about the service and staff friendliness at Parma Cucina Italiana?
List(),What is the best recipe for homemade lasagna?
List(),Can you summarize reviews for a dentist in New York?


### 8.3. Convert Questions to MLflow Evaluation Format

In [0]:
# Convert Section 8 evaluation questions into MLflow GenAI evaluation format
# MLflow requires expectations to be a dictionary

eval_data = []

for item in EVALUATION_QUESTIONS:
    expected_tools_text = ", ".join(item.get("expected_tools", []))

    if expected_tools_text == "":
        expected_tools_text = "none; the agent should reject the request if it is out of scope"

    eval_data.append(
        {
            "inputs": {
                "question": item["question"]
            },
            "expectations": {
                "expected_tools": expected_tools_text,
                "expected_behavior": (
                    "The agent should answer the question using the appropriate restaurant analysis tools, "
                    "stay grounded in retrieved reviews, business metadata, or tool outputs, "
                    "provide specific and useful business insight, and reject the request gracefully if it is out of scope."
                )
            }
        }
    )

print("=" * 60)
print("Evaluation dataset created...")
print("=" * 60)
print(f"Total evaluation records: {len(eval_data)}")

# Do not use display(eval_data) here because Databricks may not infer nested schemas correctly
for row in eval_data:
    print("\nQuestion:")
    print(row["inputs"]["question"])
    print("Expected tools:")
    print(row["expectations"]["expected_tools"])
    print("Expected behavior:")
    print(row["expectations"]["expected_behavior"])

Evaluation dataset created...
Total evaluation records: 5

Question:
How many reviews does Parma Cucina Italiana have, and what is its average rating?
Expected tools:
business_lookup
Expected behavior:
The agent should answer the question using the appropriate restaurant analysis tools, stay grounded in retrieved reviews, business metadata, or tool outputs, provide specific and useful business insight, and reject the request gracefully if it is out of scope.

Question:
Compare Parma Cucina Italiana and Cesarina. What does Cesarina appear to do better?
Expected tools:
competitor_comparison
Expected behavior:
The agent should answer the question using the appropriate restaurant analysis tools, stay grounded in retrieved reviews, business metadata, or tool outputs, provide specific and useful business insight, and reject the request gracefully if it is out of scope.

Question:
What are customers saying about the service and staff friendliness at Parma Cucina Italiana?
Expected tools:
them

### 8.4. Create Prediction Function

In [0]:
def predict_fn(question: str) -> str:
    """
    Run the default restaurant agent on a single evaluation question.
    The default agent uses Llama 3.3 70B.
    """
    
    response = AGENT.predict(
        question,
        verbose=False
    )
    
    return response

print("Prediction function created.")

Prediction function created.


In [0]:
## quick test
predict_fn("How many reviews does Parma Cucina Italiana have, and what is its average rating?")

'Parma Cucina Italiana has an average rating of 4.67 and 552 reviews.'

[Trace(trace_id=tr-07add3542c532122c8bce620111f0c38), Trace(trace_id=tr-92cebc2322a23c3c408aa15b9ae6d88f)]

### 8.5. Create T/F LLM Judge

In [0]:
from mlflow.genai.judges import make_judge

restaurant_owner_judge = make_judge(
    name="restaurant_owner_judge",
    instructions="""
You are evaluating a restaurant-review analysis agent for a final project.

The agent's purpose is to help a small restaurant owner understand San Diego Italian restaurant performance using:
- business metadata,
- rating metrics,
- rating distributions,
- retrieved review evidence,
- competitor comparison outputs,
- and tool outputs from the agent.

User input:
{{ inputs }}

Agent response:
{{ outputs }}

Expected behavior:
{{ expectations }}

Trace:
{{ trace }}

You must evaluate the agent response using the criteria below.

Return TRUE only if all required criteria are satisfied.
Return FALSE if any required criterion is not satisfied.

Criteria:

1. Groundedness:
Is the response grounded in the retrieved reviews, business metadata, or tool outputs?
True if the response does not invent unsupported facts or make claims that cannot be traced to the available evidence.

2. Answer relevance:
Does the response answer the user’s actual question?
True if the response directly addresses the user’s request rather than giving a generic or unrelated answer.

3. Restaurant-owner usefulness:
Is the response useful for a restaurant owner?
True if the response provides practical insight, decision support, or a clear next step.

4. Specificity:
Is the response specific enough to be meaningful?
True if the response includes concrete themes, complaints, strengths, comparisons, or recommendations rather than vague statements.

5. Appropriate tool use:
Did the agent use the appropriate tool or tool sequence?
True if the tool use matches the query type, such as lookup for metadata, retrieval for review evidence, or competitor comparison for comparison questions.

6. Out-of-scope handling:
If the input was irrelevant or out of scope, did the agent reject it gracefully?
True if the agent politely refused or redirected the user to an appropriate restaurant-review analysis task.
For in-scope questions, treat this criterion as satisfied.

7. Clarity:
Is the response clear and understandable for a small business owner?
True if the response is organized, concise, and written in plain business language.

Final decision rule:
- Return TRUE if all applicable criteria are satisfied.
- Return FALSE if any applicable criterion fails.
- Return only TRUE or FALSE.
- Do not include explanations, markdown, or extra text.
""",
    model="databricks:/databricks-gpt-oss-20b",
    feedback_value_type=bool
)

print("Custom TRUE/FALSE judge created: restaurant_owner_judge")

Custom TRUE/FALSE judge created: restaurant_owner_judge


### 8.6. Run LLM Judge Evaluation

In [0]:
print("=" * 60)
print("Running LLM judge evaluation...")
print("=" * 60)

eval_results = mlflow.genai.evaluate(
    data=eval_data,
    predict_fn=predict_fn,
    scorers=[
        restaurant_owner_judge
    ]
)

print("\nEvaluation complete.")
print("Review the MLflow evaluation results in the Experiments UI.")

Running LLM judge evaluation...


2026/06/17 04:02:35 INFO mlflow.genai.utils.data_validation: Testing model prediction with the first sample in the dataset. To disable this check, set the MLFLOW_GENAI_EVAL_SKIP_TRACE_VALIDATION environment variable to True.


Evaluating:   0%|          | 0/5 [Elapsed: 00:00, Remaining: ?]

[NOTICE] Using a notebook authentication token. Recommended for development only. For improved performance, please use Service Principal based authentication. To disable this message, pass disable_notice=True.
[NOTICE] Using a notebook authentication token. Recommended for development only. For improved performance, please use Service Principal based authentication. To disable this message, pass disable_notice=True.


2026/06/17 04:02:54 WARNING mlflow.genai.evaluation.harness: Some scorer invocations failed during evaluation. Failure summary: 'restaurant_owner_judge': 3/5 failed. Check individual trace assessments for detailed error messages.



Evaluation complete.
Review the MLflow evaluation results in the Experiments UI.
